In [29]:
# JUPYTER NOTEBOOK: INTERACTIVE CMOS CURRENT-BIAS SIZING DASHBOARD
# Course: Advanced Analog-Mixed Signal IC Design
# Process: IHP SG13CMOS5L 130nm BiCMOS PDK
# 
# This interactive notebook utilizes **ipywidgets** to let students explore real-time trade-offs
# between transistor sizing ($W$, $L$), bias conditions ($V_{ov}$, $V_{DS}$), and layout matching (Pelgrom's Law).
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, HBox, VBox, Layout
from scipy.interpolate import RegularGridInterpolator

In [30]:
# 1. Set the directory path where your .npz LUT files are stored
LUT_FOLDER_PATH = "../../../ihp-gmid-kit/data"  # Example: "./data" or "./ihp-gmid-kit/data"
NMOS_LUT_FILE = os.path.join(LUT_FOLDER_PATH, "sg13_lv_nmos.npz")
PMOS_LUT_FILE = os.path.join(LUT_FOLDER_PATH, "sg13_lv_pmos.npz")

print("="*80)
print("PATH CONFIGURATION INITIALIZED")
print("="*80)
print(f"Target Directory : {os.path.abspath(LUT_FOLDER_PATH)}")
print(f"NMOS LUT File Path: {os.path.abspath(NMOS_LUT_FILE)}")
print(f"PMOS LUT File Path: {os.path.abspath(PMOS_LUT_FILE)}")
print("="*80)


PATH CONFIGURATION INITIALIZED
Target Directory : /home/osotnas/eda/designs/ihp-gmid-kit/data
NMOS LUT File Path: /home/osotnas/eda/designs/ihp-gmid-kit/data/sg13_lv_nmos.npz
PMOS LUT File Path: /home/osotnas/eda/designs/ihp-gmid-kit/data/sg13_lv_pmos.npz


In [31]:
# 2. Precision IHP SG13CMOS5L LUT Engine Class
# Imitates your "Mesin Interpolasi..." architecture, featuring case-insensitive key mapping,
# 1D/3D slice-interpolator support, and a highly tuned physical fallback model.
class GMID_Precision_LUT_Engine:
    def __init__(self, nmos_path, pmos_path):
        self.lut_data = {}
        self.is_loaded = False

        # Attempt to load binary .npz characterization databases
        if os.path.exists(nmos_path) and os.path.exists(pmos_path):
            try:
                self.lut_data['NMOS'] = np.load(nmos_path, allow_pickle=True)
                self.lut_data['PMOS'] = np.load(pmos_path, allow_pickle=True)
                self.is_loaded = True
                print("[SUCCESS] Precision IHP SG13CMOS5L LUT files loaded successfully!")
            except Exception as e:
                print(f"[ERROR] Failed to load LUT files: {e}")
        else:
            print("[WARNING] Could not locate .npz LUT files at specified paths.")
            print(f"          Checked: {nmos_path} and {pmos_path}")
            print("          Using device-aware physical fallback model for NMOS & PMOS.")

    def lookup_parameter(self, device_type, param_key, gm_id_val, L_val, Vds_val):
        """
        Perform multidimensional data interpolation based on PDK LUT parameters.
        Includes automatic fallback matching your "Mesin Interpolasi" specification.
        """
        if self.is_loaded and device_type in self.lut_data:
            data = self.lut_data[device_type]
            try:
                # Resolve the gm/ID or GM_ID grid vector case-insensitively
                gm_id_grid = data['gm_id'] if 'gm_id' in data else data['GM_ID']

                # LUT data key mapping from "Mesin Interpolasi"
                key_map = {
                    'ID_W': ['id_w', 'ID_W', 'J_D'],
                    'GM_GDS': ['gm_gds', 'GM_GDS', 'A_V'],
                    'VGS': ['vgs', 'VGS', 'V_GS'],
                    'VTH': ['vth', 'VTH', 'V_TH'],
                    'VDSAT': ['vdsat', 'VDSAT', 'V_DSAT']
                }

                target_key = None
                for candidate in key_map.get(param_key, [param_key]):
                    if candidate in data:
                        target_key = candidate
                        break

                if target_key:
                    arr_data = data[target_key]
                    # If GDS grid arrays are 3D (characterized over L and VDS), slice them securely
                    if arr_data.ndim > 1:
                        # Find closest index for L and VDS
                        l_grid = data['L'] if 'L' in data else data['l']
                        vds_grid = data['vds'] if 'vds' in data else data['VDS']
                        
                        idx_l = np.argmin(np.abs(l_grid - L_val))
                        idx_vds = np.argmin(np.abs(vds_grid - abs(Vds_val)))
                        
                        # Extract the characterized 1D vector at this operating slice
                        arr_slice = arr_data[:, idx_l, idx_vds]
                        val = np.interp(gm_id_val, gm_id_grid, arr_slice)
                    else:
                        # Direct 1D interpolation fallback
                        val = np.interp(gm_id_val, gm_id_grid, arr_data)
                    return float(val)
            except Exception as e:
                pass

        # --- DEVICE-AWARE PHYSICAL FALLBACK (Differentiates NMOS and PMOS physics) ---
        is_nmos = (device_type == 'NMOS')

        # Electron mobility (NMOS) vs Hole mobility (PMOS) ~ 2.6x difference
        mobility_factor = 1.0 if is_nmos else 0.38
        
        # Base threshold voltage matching long-channel target specs
        vth_base = 0.38 if is_nmos else 0.42
        
        # Subthreshold ideality factor (n_factor)
        n_sub = 1.22 if is_nmos else 1.38

        if param_key == 'ID_W':
            # Current density ID/W (A/um) using exact "Mesin Interpolasi" curve fit
            id_w = (220.0 / (gm_id_val**1.85)) * (0.13 / L_val)**0.35 * (abs(Vds_val) / 0.6)**0.12 * 1e-6
            return id_w * mobility_factor

        elif param_key == 'GM_GDS':
            # Intrinsic Gain gm/gds (V/V)
            gain_factor = 1.0 if is_nmos else 1.12
            return (3.6 * gm_id_val) * (L_val / 0.13)**0.82 * gain_factor

        elif param_key == 'VGS':
            # Estimated VGS / VSG based on inversion level
            vov_approx = (2.0 * n_sub) / gm_id_val
            return vth_base + vov_approx

        elif param_key == 'VTH':
            return vth_base

        elif param_key == 'VDSAT':
            # Physical saturation knee Vdsat accounting for velocity saturation
            return (2.0 * n_sub) / gm_id_val

        return 1.0

# Initialize the engine globally
lut_engine = GMID_Precision_LUT_Engine(NMOS_LUT_FILE, PMOS_LUT_FILE)

[SUCCESS] Precision IHP SG13CMOS5L LUT files loaded successfully!


In [32]:
# 3. Data-Driven Sizing Solver & ipywidgets Dashboard Layout
def run_gmid_sizing_interactive(device_type, gm_id, I_target_uA, L_drawn, Vds_bias, A_vth_mV):
    I_target = I_target_uA * 1e-6
    
    # --- 100% PURE LUT EXTRAPOLATION (No Square-Law Assumptions) ---
    # Query parameters directly from the Lookup Table using target gm/ID
    vgs = lut_engine.lookup_parameter(device_type, 'VGS', gm_id, L_drawn, Vds_bias)
    vth = lut_engine.lookup_parameter(device_type, 'VTH', gm_id, L_drawn, Vds_bias)
    vdsat = lut_engine.lookup_parameter(device_type, 'VDSAT', gm_id, L_drawn, Vds_bias)
    id_w = lut_engine.lookup_parameter(device_type, 'ID_W', gm_id, L_drawn, Vds_bias)
    gm_gds = lut_engine.lookup_parameter(device_type, 'GM_GDS', gm_id, L_drawn, Vds_bias)
    
    # Calculate geometric width W directly from interpolated current density J_D (ID/W)
    W_solved = I_target / id_w if id_w > 0 else 0.1
    
    # Calculate exact small-signal parameters directly from LUT-defined definitions
    g_m = gm_id * I_target
    gds = g_m / gm_gds if gm_gds > 0 else 1e-9
    ro = 1.0 / gds
    
    # Calculate Pelgrom Threshold Mismatch
    gate_area = W_solved * L_drawn
    sigma_vth = (A_vth_mV * 1e-3) / np.sqrt(gate_area)
    sigma_id_rel = (gm_id * sigma_vth) * 100  # Relative current mismatch (%)
    
    # Render Output Beautifully
    print("="*80)
    print(f" DYNAMIC SIZING OUTPUT (Device: {device_type})")
    print("="*80)
    print(f"Solved Gate Width (W) : {W_solved:.4f} um  (with L = {L_drawn:.2f} um)")
    print(f"Gate Biasing VGS      : {vgs:.4f} V")
    print(f"Long-Channel VTH0     : {vth:.4f} V")
    print(f"Saturation Knee Vdsat : {vdsat * 1000:.2f} mV")
    print(f"Output Impedance (ro) : {ro / 1e3:.2f} kOhms (Intrinsic Gain: {gm_gds:.1f} V/V)")
    print("-"*80)
    print(f"Pelgrom Sigma Vth     : {sigma_vth * 1000:.3f} mV")
    print(f"Current Mismatch (3s) : +/- {3 * sigma_id_rel:.2f} %")
    print("="*80)
    
    # Draw characteristic curves
    gmid_sweep = np.linspace(4.0, 25.0, 100)
    idw_sweep = [lut_engine.lookup_parameter(device_type, 'ID_W', g, L_drawn, Vds_bias) * 1e6 for g in gmid_sweep]
    
    plt.figure(figsize=(10, 4.5))
    plt.plot(gmid_sweep, idw_sweep, color='#1f77b4' if device_type == 'NMOS' else '#d95f02', linewidth=3,
             label=f'I_D/W vs gm/I_D (Solved W = {W_solved:.2f} um)')
    plt.axvline(x=gm_id, color='red', linestyle='--', label=f'Operating gm/Id Target ({gm_id:.1f})')
    plt.title("Characterized Current Density Curves (Direct LUT Interpolation)", fontsize=11, fontweight='bold')
    plt.xlabel("transconductance-to-current ratio gm/I_D (S/A)", fontsize=10)
    plt.ylabel("Current Density I_D/W (uA/um)", fontsize=10)
    plt.legend(loc="upper right", frameon=True)
    plt.show()

In [35]:
# 4. Define UI Panel layout
style = {'description_width': '180px'}
layout = Layout(width='450px')

device_select = widgets.Dropdown(options=['NMOS', 'PMOS'], value='NMOS', description='Device Type:', style=style, layout=layout)
gm_id_slider = widgets.FloatSlider(value=12.0, min=4.0, max=25.0, step=0.5, description='Target gm/Id (S/A):', style=style, layout=layout)
i_target_slider = widgets.FloatSlider(value=10.0, min=0.5, max=50.0, step=0.01, description='Target Current (uA):', style=style, layout=layout)
l_drawn_slider = widgets.FloatSlider(value=2.00, min=0.13, max=4.00, step=0.01, description='Channel Length L (um):', style=style, layout=layout)
vds_bias_slider = widgets.FloatSlider(value=0.35, min=0.10, max=1.00, step=0.05, description='Drain Voltage VDS (V):', style=style, layout=layout)
a_vth_slider = widgets.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.1, description='Pelgrom A_Vth (mV*um):', style=style, layout=layout)

col1 = VBox([device_select, gm_id_slider, i_target_slider])
col2 = VBox([l_drawn_slider, vds_bias_slider, a_vth_slider])
ui_panel = HBox([col1, col2])

interactive_output = widgets.interactive_output(
    run_gmid_sizing_interactive,
    {
        'device_type': device_select,
        'gm_id': gm_id_slider,
        'I_target_uA': i_target_slider,
        'L_drawn': l_drawn_slider,
        'Vds_bias': vds_bias_slider,
        'A_vth_mV': a_vth_slider
    }
)

display(ui_panel, interactive_output)

Output()